# Einzelner Airy-Spot: Atom-Versatz gegen Waist

Zwei Karten ueber derselben Ebene: **y = Waist $\omega$**, **x = Versatz $r$ des
Atoms aus der Spot-Mitte**. Aufgetragen sind die atomgewichtete Uniformity
$U_w$ und der atomgewichtete Crosstalk $\eta_w$. Gerechnet wird mit
`lib/single_beam.py` (`weighted_metrics`), hier steht nur, was eingetragen,
gezeichnet und gespeichert wird.

Die gestrichelte Linie ist $r = \omega$ - weiter als eine Waist aus der Mitte
ist das Atom nicht mehr sinnvoll gefangen.

Ausgabe in `Notebook_Ergebnisse/`: die Karten, ein Schnitt bei fester Waist,
die Daten als CSV und die Kennzahlen als Textdatei.

In [1]:
# ====================  PARAMETER  ====================================
WAIST_VON_UM, WAIST_BIS_UM = 0.75, 2.5   # y-Achse (0.75 um = 1.784 mm vor der Linse)
SHIFT_VON_UM, SHIFT_BIS_UM = 0.0, 2.5    # x-Achse, Versatz aus der Mitte
N_WAIST, N_SHIFT           = 61, 61      # Rasterpunkte (61x61 ~ 2 min)

SCHNITT_OMEGA_UM = 1.9     # zusaetzlicher Schnitt bei dieser Waist
N_SCHNITT        = 201     # Stuetzstellen des Schnitts (eigene, feinere Rechnung)
SCHNITT_R_MAX_UM = None    # Ende des Schnitts; None = bis r = omega

PROFIL        = "airy"
AIRY_FAKTOR   = 1.482951   # first_zero_radius = Faktor * waist
PITCH_UM      = 5.288      # Fallenabstand (fuer den Crosstalk)
RICHTUNG      = "x"        # "x" horizontal, "y" vertikal

ATOM_T_UK     = 17.0       # Atomtemperatur (uK)  -> sigma_atom
NU_R_KHZ      = 60.4       # radiale Fallenfrequenz (kHz)

GRENZLINIE    = True       # Linie r = omega (Versatz = Waist) einzeichnen
NAME          = "SingleSpot_Shift_vs_Waist"
# =====================================================================

import sys
from pathlib import Path
BASE = Path.cwd()                 # Ordner Combinated_Optimization
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))
OUT = BASE / "Notebook_Ergebnisse"
OUT.mkdir(exist_ok=True)
print("Ausgabe nach:", OUT)

Ausgabe nach: /mnt/user-data/uploads/Multitone_Simulation/Amplitudes/Combinated_Optimization/Notebook_Ergebnisse


In [2]:
# ====================  RECHNEN  ======================================
import time
import numpy as np
from lib import single_beam as sb

params = dict(sb.DEFAULTS, profile=PROFIL, airy_scale_factor=AIRY_FAKTOR,
              pitch=PITCH_UM * 1e-6,
              atom_temperature=ATOM_T_UK * 1e-6, trap_freq_r=NU_R_KHZ * 1e3)
sigma = sb.sigma_thermal(params["atom_mass"], 2 * np.pi * params["trap_freq_r"],
                         params["atom_temperature"])
print(f"sigma_atom = {sigma*1e9:.1f} nm")


def metriken(omega_um, r_um_wert):
    """U_w und eta_w in Prozent bei Waist und Versatz (beide in um)."""
    dx, dy = (r_um_wert, 0.0) if RICHTUNG == "x" else (0.0, r_um_wert)
    q = dict(params, atom_offset_x=dx * 1e-6, atom_offset_y=dy * 1e-6)
    u, e = sb.weighted_metrics(omega_um * 1e-6, q, sigma_atom=sigma)
    return u * 100.0, e * 100.0


waist_um = np.linspace(WAIST_VON_UM, WAIST_BIS_UM, N_WAIST)
shift_um = np.linspace(SHIFT_VON_UM, SHIFT_BIS_UM, N_SHIFT)
U_w = np.full((N_WAIST, N_SHIFT), np.nan)
eta_w = np.full((N_WAIST, N_SHIFT), np.nan)

t0 = time.perf_counter()
for i, w in enumerate(waist_um):
    for j, r in enumerate(shift_um):
        U_w[i, j], eta_w[i, j] = metriken(w, r)
    if (i + 1) % 10 == 0 or i + 1 == N_WAIST:
        print(f"  Zeile {i+1}/{N_WAIST}")
print(f"fertig in {time.perf_counter()-t0:.1f} s")

sigma_atom = 107.6 nm


  Zeile 10/61


  Zeile 20/61


  Zeile 30/61


  Zeile 40/61


  Zeile 50/61


  Zeile 60/61


  Zeile 61/61
fertig in 122.6 s


In [3]:
# ====================  PLOT: die beiden Karten  ======================
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

FIG_W_IN, PT = 16.0 / 2.54, 10.0          # 16 cm = \textwidth, 10 pt Fliesstext
LBL_X = r"$r$ ($\mathrm{\mu}$m)"
LBL_Y = r"$\omega$ ($\mathrm{\mu}$m)"

# Masse in cm statt tight_layout: sonst bleibt zwischen den beiden Haelften
# ein breites weisses Band stehen und die Karten werden unnoetig klein.
W_CM   = FIG_W_IN * 2.54
L_CM   = 1.50     # y-Beschriftung und Ticks
CB_GAP = 0.12     # Luft zwischen Karte und Farbskala
CB_W   = 0.20     # Breite der Farbskala
CB_LBL = 1.72     # Ticks und Beschriftung der Farbskala
MITTE  = 0.40     # Abstand der beiden Haelften
BOT_CM, TOP_CM = 1.25, 0.15
P_CM = (W_CM - 2.0 * (L_CM + CB_GAP + CB_W + CB_LBL) - MITTE) / 2.0   # Kantenlaenge
H_CM = BOT_CM + P_CM + TOP_CM
BLOCK = L_CM + P_CM + CB_GAP + CB_W + CB_LBL

kante = [shift_um[0], shift_um[-1], waist_um[0], waist_um[-1]]
# Endpunkte der Linie r = omega innerhalb beider Achsen
g = np.linspace(max(shift_um[0], waist_um[0]), min(shift_um[-1], waist_um[-1]), 2)

fig = plt.figure(figsize=(FIG_W_IN, H_CM / 2.54))
achsen = []
for k, (Z, lab, cmap) in enumerate(((U_w,   r"$U_w$ (%)",    "viridis"),
                                    (eta_w, r"$\eta_w$ (%)", "magma"))):
    x0 = k * (BLOCK + MITTE) + L_CM
    ax = fig.add_axes([x0 / W_CM, BOT_CM / H_CM, P_CM / W_CM, P_CM / H_CM])
    cax = fig.add_axes([(x0 + P_CM + CB_GAP) / W_CM, BOT_CM / H_CM,
                        CB_W / W_CM, P_CM / H_CM])
    im = ax.imshow(Z, origin="lower", extent=kante, aspect="auto", cmap=cmap,
                   norm=LogNorm(vmin=np.nanmin(Z), vmax=np.nanmax(Z)))
    if GRENZLINIE:
        # r = omega: weiter aussen ist das Atom nicht mehr sinnvoll gefangen
        ax.plot(g, g, color="w", lw=1.2, ls="--")
    ax.set_xlabel(LBL_X)
    ax.set_ylabel(LBL_Y)
    cb = fig.colorbar(im, cax=cax)
    cb.set_label(lab)
    achsen.append(ax)

for o in fig.findobj(plt.Text):
    o.set_fontsize(PT)
for ax in fig.axes:
    ax.tick_params(labelsize=PT)

# Beschriftung der Linie: der Winkel steht erst nach dem Zeichnen fest, die
# Achsen sind ja nicht gleich skaliert - 45 Grad waeren falsch.
fig.canvas.draw()
if GRENZLINIE:
    for ax in achsen:
        p0 = ax.transData.transform((g[0], g[0]))
        p1 = ax.transData.transform((g[-1], g[-1]))
        winkel = np.degrees(np.arctan2(p1[1] - p0[1], p1[0] - p0[0]))
        mitte = g[0] + 0.60 * (g[-1] - g[0])
        ax.text(mitte, mitte, r"$r = \omega$", color="w", fontsize=PT,
                rotation=winkel, rotation_mode="anchor", ha="center", va="bottom")

pdf = OUT / f"{NAME}_{PROFIL}.pdf"
fig.savefig(pdf)
print(f"gespeichert: {pdf.name}   ({W_CM:.1f} x {H_CM:.2f} cm, "
      f"Karten {P_CM:.2f} x {P_CM:.2f} cm)")

gespeichert: SingleSpot_Shift_vs_Waist_airy.pdf   (16.0 x 5.66 cm, Karten 4.26 x 4.26 cm)


In [4]:
# ====================  PLOT: Schnitt bei fester Waist  ===============
# Beide Kurven in EINEM Plot, lineare Achse - bis r = omega bleiben sie in
# derselben Groessenordnung, da braucht es keine Logachse und keine zwei
# Teilbilder.
r_max_s = SCHNITT_OMEGA_UM if SCHNITT_R_MAX_UM is None else SCHNITT_R_MAX_UM
r_s = np.linspace(SHIFT_VON_UM, r_max_s, N_SCHNITT)
U_s = np.empty(N_SCHNITT)
E_s = np.empty(N_SCHNITT)
for j, r in enumerate(r_s):
    U_s[j], E_s[j] = metriken(SCHNITT_OMEGA_UM, r)

fig2, ax = plt.subplots(figsize=(FIG_W_IN, 0.48 * FIG_W_IN))
ax.plot(r_s, U_s, color="#009E73", lw=1.6, label=r"$U_w$")
ax.plot(r_s, E_s, color="#882255", lw=1.6, label=r"$\eta_w$")
ax.set_xlabel(LBL_X)
ax.set_ylabel(r"$U_w$,  $\eta_w$ (%)")
ax.set_xlim(r_s[0], r_s[-1])
ax.set_ylim(bottom=0.0)
ax.grid(alpha=0.25, lw=0.5)
ax.legend(loc="upper left", framealpha=0.9, edgecolor="0.8")
# Zweite Achse: Versatz in Einheiten der Waist, 1 ist die Grenze r = omega
sek = ax.secondary_xaxis("top", functions=(lambda v: v / SCHNITT_OMEGA_UM,
                                           lambda v: v * SCHNITT_OMEGA_UM))
sek.set_xlabel(r"$r\,/\,\omega$")

for o in fig2.findobj(plt.Text):
    o.set_fontsize(PT)
for a in fig2.axes:
    a.tick_params(labelsize=PT)
fig2.tight_layout()

pdf2 = OUT / f"{NAME}_schnitt_omega{SCHNITT_OMEGA_UM:.2f}um.pdf"
fig2.savefig(pdf2)
csv2 = pdf2.with_suffix(".csv")
with open(csv2, "w", encoding="utf-8") as f:
    f.write(f"# Schnitt bei omega = {SCHNITT_OMEGA_UM} um, {PROFIL}, "
            f"Airy-Faktor {AIRY_FAKTOR}, pitch {PITCH_UM} um, Richtung {RICHTUNG}\n"
            f"# sigma_atom = {sigma*1e9:.1f} nm\n"
            "shift_um,U_w_prozent,eta_w_prozent\n")
    np.savetxt(f, np.column_stack([r_s, U_s, E_s]), delimiter=",", fmt="%.6g")
print("gespeichert:", pdf2.name, "und", csv2.name)

gespeichert: SingleSpot_Shift_vs_Waist_schnitt_omega1.90um.pdf und SingleSpot_Shift_vs_Waist_schnitt_omega1.90um.csv


In [5]:
# ====================  DATEN SPEICHERN  ==============================
W, R = np.meshgrid(waist_um, shift_um, indexing="ij")
tab = np.column_stack([W.ravel(), R.ravel(), U_w.ravel(), eta_w.ravel()])

csv = OUT / f"{NAME}_{PROFIL}.csv"
with open(csv, "w", encoding="utf-8") as f:
    f.write(f"# Einzelner Spot, {PROFIL}, Airy-Faktor {AIRY_FAKTOR}, "
            f"pitch {PITCH_UM} um, Richtung {RICHTUNG}\n"
            f"# sigma_atom = {sigma*1e9:.1f} nm "
            f"(T = {ATOM_T_UK} uK, nu_r = {NU_R_KHZ} kHz)\n"
            "waist_um,shift_um,U_w_prozent,eta_w_prozent\n")
    np.savetxt(f, tab, delimiter=",", fmt="%.6g")
print("gespeichert:", csv.name, f"({tab.shape[0]} Zeilen)")

gespeichert: SingleSpot_Shift_vs_Waist_airy.csv (3721 Zeilen)


In [6]:
# ====================  KENNZAHLEN  ===================================
d_w = waist_um[1] - waist_um[0]
d_r = shift_um[1] - shift_um[0]
zelle = 2.0 * params["weighted_n_sigma"] * sigma * 1e9 / (params["weighted_n_grid"] - 1)
innen = shift_um[None, :] <= waist_um[:, None]          # Bereich r <= omega
j0 = int(np.argmin(np.abs(shift_um)))                   # Spalte r = 0

z = ["RECHENRASTER",
     f"  Karte          {N_WAIST} x {N_SHIFT} Punkte",
     f"  d_omega        {d_w*1000:.1f} nm   ({WAIST_VON_UM} .. {WAIST_BIS_UM} um)",
     f"  d_r            {d_r*1000:.1f} nm   ({SHIFT_VON_UM} .. {SHIFT_BIS_UM} um)",
     f"  Schnitt        {N_SCHNITT} Punkte, d_r = {(r_s[1]-r_s[0])*1000:.1f} nm",
     f"  sigma_atom     {sigma*1e9:.1f} nm",
     f"  Sub-Gitter     {params['weighted_n_grid']} x {params['weighted_n_grid']} "
     f"ueber +-{params['weighted_n_sigma']} sigma  ->  Zelle {zelle:.1f} nm",
     "", "EXTREMWERTE (ganze Karte)"]
for name, Z in (("U_w  ", U_w), ("eta_w", eta_w)):
    for wie, idx in (("min", np.nanargmin(Z)), ("max", np.nanargmax(Z))):
        i, j = np.unravel_index(idx, Z.shape)
        z.append(f"  {name} {wie}   {Z[i,j]:10.4g} %   bei omega = {waist_um[i]:.3f} um, "
                 f"r = {shift_um[j]:.3f} um")
z += ["", "INNERHALB DER GRENZE  r <= omega"]
for name, Z in (("U_w  ", U_w), ("eta_w", eta_w)):
    Zi = np.where(innen, Z, np.nan)
    for wie, idx in (("min", np.nanargmin(Zi)), ("max", np.nanargmax(Zi))):
        i, j = np.unravel_index(idx, Z.shape)
        z.append(f"  {name} {wie}   {Zi[i,j]:10.4g} %   bei omega = {waist_um[i]:.3f} um, "
                 f"r = {shift_um[j]:.3f} um")
z += ["", "OHNE VERSATZ  (r = 0)"]
for name, Z in (("U_w  ", U_w), ("eta_w", eta_w)):
    s = Z[:, j0]
    z.append(f"  {name} min   {np.nanmin(s):10.4g} %   bei omega = {waist_um[np.nanargmin(s)]:.3f} um")
    z.append(f"  {name} max   {np.nanmax(s):10.4g} %   bei omega = {waist_um[np.nanargmax(s)]:.3f} um")
z += ["", f"SCHNITT BEI  omega = {SCHNITT_OMEGA_UM} um"]
for name, s in (("U_w  ", U_s), ("eta_w", E_s)):
    z.append(f"  {name} bei r = 0        {s[0]:10.4g} %")
    z.append(f"  {name} bei r = omega    {np.interp(SCHNITT_OMEGA_UM, r_s, s):10.4g} %")
    z.append(f"  {name} max              {s.max():10.4g} %   bei r = {r_s[int(s.argmax())]:.3f} um")

text = "\n".join(z)
print(text)
kenn = OUT / f"{NAME}_{PROFIL}_kennzahlen.txt"
kenn.write_text(text + "\n", encoding="utf-8")
print("\ngespeichert:", kenn.name)

RECHENRASTER
  Karte          61 x 61 Punkte
  d_omega        29.2 nm   (0.75 .. 2.5 um)
  d_r            41.7 nm   (0.0 .. 2.5 um)
  Schnitt        201 Punkte, d_r = 9.5 nm
  sigma_atom     107.6 nm
  Sub-Gitter     241 x 241 ueber +-6 sigma  ->  Zelle 5.4 nm

EXTREMWERTE (ganze Karte)
  U_w   min       0.6163 %   bei omega = 2.500 um, r = 0.000 um
  U_w   max        180.3 %   bei omega = 0.750 um, r = 1.083 um
  eta_w min      0.04526 %   bei omega = 0.838 um, r = 0.000 um
  eta_w max         1640 %   bei omega = 1.683 um, r = 2.500 um

INNERHALB DER GRENZE  r <= omega
  U_w   min       0.6163 %   bei omega = 2.500 um, r = 0.000 um
  U_w   max        65.47 %   bei omega = 0.750 um, r = 0.750 um
  eta_w min      0.04526 %   bei omega = 0.838 um, r = 0.000 um
  eta_w max        76.11 %   bei omega = 2.500 um, r = 2.500 um

OHNE VERSATZ  (r = 0)
  U_w   min       0.6163 %   bei omega = 2.500 um
  U_w   max        6.579 %   bei omega = 0.750 um
  eta_w min      0.04526 %   bei omega = 0.

## Notizen